# Threat-Aware LCD: full publication run (Colab GPU)

This notebook reproduces every table and figure of the paper with the **ResNet-18** configuration:

| Step | What runs | Approx. time on a T4 |
|---|---|---|
| 1-3 | setup, code, data | 5 min |
| 4 | train 3 CIFAR-10 seeds, 3 SVHN seeds, BadNets 1/5/10% | 2-3 h |
| 5 | `pub_eval.py --full` for every clean model | 3-5 h |
| 6 | backdoor track (AC + Neural Cleanse) | 30-45 min |
| 7 | aggregate tables, figures and `numbers.tex` | 1 min |

Everything is written to Google Drive, and every step **skips work that is already finished**, so if Colab disconnects
just run all cells again. Choose *Runtime → Change runtime type → GPU* first.

## 1. Environment

In [ ]:
!nvidia-smi -L
!pip -q install git+https://github.com/fra31/auto-attack
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Drive folder and code
The folder layout mirrors the project: `04_Practical_Implementation/lcd_core.py` and `08_Publication/*.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/ThreatAwareLCD_publication'
for d in ['04_Practical_Implementation', '08_Publication/checkpoints', '08_Publication/results', '08_Publication/logs',
          '08_Publication/paper/tables', '08_Publication/paper/figures', 'data']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
%cd {ROOT}/08_Publication

In [ ]:
%%writefile {ROOT}/04_Practical_Implementation/lcd_core.py
"""
Threat-Aware LCD (ARMOR-inspired) - core implementation.

Pallavi Sarangi (M25DE2073), M.Tech Data Engineering, IIT Jodhpur
Supervisor: Dr. Nitin Niranjan Awathare

Everything the architecture needs lives here so that the local pilot (run_pilot.py) and the
Colab notebook use exactly the same code:

    models        ResNet-18 (MTP-1 setting) and ResNet-20 (fast CPU pilot), with input normalisation inside
    squeeze S     3-bit colour depth + 2x2 median filter                 (Xu et al., NDSS 2018)
    signals       p0, ps, D = JS/ln2, U = max normalised entropy, TS      (architecture block 3)
    router        fast path / Stage 2 / abstain                           (architecture block 4)
    Stage 2       N random resize+pad views of S(x), averaged             (Xie et al., ICLR 2018)
    calibration   temperature scaling + validation-only threshold search  (Guo et al., ICML 2017)
    attacks       FGSM, BIM, C&W-L2, adaptive BPDA+EOT, BadNets
    audit         activation clustering for the poisoned model             (Chen et al., 2019)

All images are tensors in pixel space [0, 1]; the model normalises internally, so attacks and
defences never have to think about CIFAR mean/std.
"""
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
K = 10


# =============================================================================================
# Models
# =============================================================================================
class Normalize(nn.Module):
    def __init__(self, mean=CIFAR_MEAN, std=CIFAR_STD):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std


class _BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.short = nn.Sequential()
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(out)) + self.short(x))


class ResNet20(nn.Module):
    """CIFAR ResNet-20 (He et al., 2016, Sec. 4.2): 3 stages x 3 blocks, 16/32/64 channels."""

    def __init__(self, num_classes=K):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, cin = [], 16
        for cout, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(3):
                layers.append(_BasicBlock(cin, cout, stride if i == 0 else 1))
                cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(64, num_classes)

    def features(self, x):
        return F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1).flatten(1)

    def forward(self, x):
        return self.fc(self.features(x))


def resnet18_cifar(num_classes=K):
    """torchvision ResNet-18 with the usual CIFAR stem (3x3 conv, no max-pool)."""
    m = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    return m


def make_model(arch="resnet18"):
    net = resnet18_cifar() if arch == "resnet18" else ResNet20()
    return nn.Sequential(Normalize(), net)


def penultimate(model, x):
    """Penultimate-layer activations (used by the activation-clustering backdoor audit)."""
    net = model[1]
    x = model[0](x)
    if isinstance(net, ResNet20):
        return net.features(x)
    feats = nn.Sequential(*list(net.children())[:-1])
    return feats(x).flatten(1)


# =============================================================================================
# Data
# =============================================================================================
def load_cifar10(root, val_size=5000, seed=2073):
    """Returns float tensors in [0,1]: (x_train, y_train), (x_val, y_val), (x_test, y_test).
    Validation is carved out of the official training split; the official test split is held out."""
    import os
    to_t = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255.0)
    cache = os.path.join(root, "cifar10.npz")      # optional local cache (uint8 HWC arrays, official order)
    if os.path.exists(cache):
        d = np.load(cache)
        x, y = to_t(d["x_train"]), torch.from_numpy(d["y_train"])
        x_test, y_test = to_t(d["x_test"]), torch.from_numpy(d["y_test"])
    else:
        tr = torchvision.datasets.CIFAR10(root, train=True, download=True)
        te = torchvision.datasets.CIFAR10(root, train=False, download=True)
        x, y = to_t(tr.data), torch.tensor(tr.targets)
        x_test, y_test = to_t(te.data), torch.tensor(te.targets)
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(x), generator=g)
    vi, ti = perm[:val_size], perm[val_size:]
    return (x[ti], y[ti]), (x[vi], y[vi]), (x_test, y_test)


def load_dataset(name, root, val_size=5000, seed=2073):
    """Generic loader: 'cifar10' (see load_cifar10) or 'svhn' (official train/test split; root must hold svhn.npz with
    uint8 HWC arrays x_train/y_train/x_test/y_test). Validation is carved out of the training split."""
    import os
    if name == "cifar10":
        return load_cifar10(root, val_size, seed)
    to_t = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255.0)
    d = np.load(os.path.join(root, f"{name}.npz"))
    x, y = to_t(d["x_train"]), torch.from_numpy(d["y_train"])
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(x), generator=g)
    vi, ti = perm[:val_size], perm[val_size:]
    return (x[ti], y[ti]), (x[vi], y[vi]), (to_t(d["x_test"]), torch.from_numpy(d["y_test"]))


def augment(x, flip=True):
    """Standard augmentation on a batch: random crop (pad 4) + horizontal flip (flip=False for digits, e.g. SVHN)."""
    n = x.shape[0]
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (n,))
    j = torch.randint(0, 9, (n,))
    ar = torch.arange(32)
    rows = (i[:, None] + ar)[:, None, :, None].expand(n, 3, 32, 40)
    out = xp.gather(2, rows)                                            # crop rows -> B,3,32,40
    cols = (j[:, None] + ar)[:, None, None, :].expand(n, 3, 32, 32)
    out = out.gather(3, cols)                                           # crop cols -> B,3,32,32
    if flip:
        f = torch.rand(n) < 0.5
        out[f] = out[f].flip(3)
    return out


def batches(x, y, bs, shuffle=False, seed=0):
    idx = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed)) if shuffle else torch.arange(len(x))
    for s in range(0, len(x), bs):
        b = idx[s:s + bs]
        yield x[b], y[b]


# =============================================================================================
# Training (clean, BadNets-poisoned, optional fast adversarial training)
# =============================================================================================
def train_model(model, x, y, epochs, bs=128, lr=0.1, wd=5e-4, device="cpu", poison_fn=None,
                adv_eps=None, log=print, seed=2073, flip=True):
    """SGD + Nesterov momentum + one-cycle LR (a short schedule that converges in few epochs).
    poison_fn(xb, yb) -> (xb, yb) injects BadNets triggers per batch (separate poisoned model).
    adv_eps enables FGSM-RS fast adversarial training (Wong et al., ICLR 2020) as a baseline."""
    torch.manual_seed(seed)
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    steps = epochs * math.ceil(len(x) / bs)
    if steps >= 20:
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.25)
    else:                                   # too few steps for a schedule (quick smoke tests only)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0)
    for ep in range(epochs):
        t0, correct, total, loss_sum = time.time(), 0, 0, 0.0
        for xb, yb in batches(x, y, bs, shuffle=True, seed=seed + ep):
            xb = augment(xb, flip)
            if poison_fn is not None:
                xb, yb = poison_fn(xb, yb)
            xb, yb = xb.to(device), yb.to(device)
            if adv_eps:
                delta = (torch.rand_like(xb) * 2 - 1) * adv_eps
                delta.requires_grad_(True)
                l_adv = F.cross_entropy(model((xb + delta).clamp(0, 1)), yb)
                g, = torch.autograd.grad(l_adv, delta)
                delta = (delta + 1.25 * adv_eps * g.sign()).clamp(-adv_eps, adv_eps).detach()
                xb = (xb + delta).clamp(0, 1)
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
        log(f"  epoch {ep + 1}/{epochs}  loss {loss_sum / total:.3f}  train-acc {correct / total:.3f}  ({time.time() - t0:.0f}s)")
    return model.eval()


@torch.no_grad()
def predict_logits(model, x, bs=500, device="cpu"):
    model.eval()
    return torch.cat([model(xb.to(device)).cpu() for xb, _ in batches(x, torch.zeros(len(x)), bs)])


def accuracy(model, x, y, device="cpu"):
    return (predict_logits(model, x, device=device).argmax(1) == y).float().mean().item()


# =============================================================================================
# Block 1 - Squeeze transform S(x): 3-bit colour depth + 2x2 median filter
# =============================================================================================
def bit_depth(x, bits=3):
    levels = 2 ** bits - 1
    return torch.round(x * levels) / levels


def median_filter(x, k=2):
    """k x k median filter (for k=2 the lower median, as scipy/Xu et al. effectively use)."""
    pad_l, pad_r = (k - 1) // 2, k // 2
    xp = F.pad(x, (pad_l, pad_r, pad_l, pad_r), mode="reflect")
    patches = xp.unfold(2, k, 1).unfold(3, k, 1)                 # B,C,H,W,k,k
    return patches.reshape(*patches.shape[:4], -1).median(dim=-1).values


def squeeze(x, bits=3, k=2):
    """S(x). Order is fixed and versioned: bit-depth reduction first, then median filter."""
    return median_filter(bit_depth(x, bits), k)


def squeeze_bpda(x, bits=3, k=2):
    """S(x) in the forward pass, identity gradient in the backward pass (BPDA, Athalye et al. 2018)."""
    return x + (squeeze(x, bits, k) - x).detach()


# =============================================================================================
# Block 2-3 - probes and threat signals
# =============================================================================================
def entropy_norm(p, eps=1e-12):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(-1) / math.log(p.shape[-1])


def js_norm(p, q, eps=1e-12):
    """Jensen-Shannon divergence divided by ln 2, so it lies in [0, 1] (Lin, 1991)."""
    p, q = p.clamp_min(eps), q.clamp_min(eps)
    m = 0.5 * (p + q)
    kl = lambda a, b: (a * (a.log() - b.log())).sum(-1)
    return ((0.5 * kl(p, m) + 0.5 * kl(q, m)) / math.log(2)).clamp(0, 1)


@torch.no_grad()
def probe(model, x, T=1.0, alpha=0.5, bits=3, k=2, device="cpu"):
    """Stage-1 probes for a batch. Returns p0, ps, D, U, TS and the fast-path prediction.
    x and S(x) are sent through the network as ONE batch of 2B images (one call, two views)."""
    x = x.to(device)
    z = model(torch.cat([x, squeeze(x, bits, k)]))
    z0, zs = z.chunk(2)
    p0, ps = F.softmax(z0 / T, -1), F.softmax(zs / T, -1)
    D = js_norm(p0, ps)
    U = torch.maximum(entropy_norm(p0), entropy_norm(ps))
    return {"p0": p0.cpu(), "ps": ps.cpu(), "D": D.cpu(), "U": U.cpu(),
            "TS": (alpha * D + (1 - alpha) * U).cpu(), "y_fast": p0.argmax(1).cpu()}


# =============================================================================================
# Stage 2 - randomised resize + pad views of S(x)
# =============================================================================================
def random_resize_pad(x, min_size=26, out=32, generator=None):
    """Xie et al.-style view: resize to r x r (r ~ U{min_size..out}), then pad at a random offset."""
    n = x.shape[0]
    r = int(torch.randint(min_size, out + 1, (1,), generator=generator))
    xr = F.interpolate(x, size=(r, r), mode="bilinear", align_corners=False)
    gap = out - r
    top = int(torch.randint(0, gap + 1, (1,), generator=generator))
    left = int(torch.randint(0, gap + 1, (1,), generator=generator))
    return F.pad(xr, (left, gap - left, top, gap - top), value=0.0)


@torch.no_grad()
def stage2(model, x, n_views=10, T=1.0, bits=3, k=2, seed=0, device="cpu"):
    """p_bar = mean_r softmax(z(T_r(S(x)))/T); U2 = H(p_bar)/ln K; D2 = mean_r JS(p_r, p_bar)/ln 2."""
    g = torch.Generator().manual_seed(seed)
    xs = squeeze(x.to(device), bits, k)
    views = torch.cat([random_resize_pad(xs, generator=g) for _ in range(n_views)])
    P = F.softmax(model(views) / T, -1).view(n_views, x.shape[0], -1)
    p_bar = P.mean(0)
    D2 = torch.stack([js_norm(P[r], p_bar) for r in range(n_views)]).mean(0)
    return {"p_bar": p_bar.cpu(), "y2": p_bar.argmax(1).cpu(), "U2": entropy_norm(p_bar).cpu(), "D2": D2.cpu()}


# =============================================================================================
# Block 4 + full LCD inference
# =============================================================================================
def lcd_predict(model, x, cfg, device="cpu", bs=250, seed=0):
    """Full Threat-Aware LCD. Returns predictions (-1 = abstain) and per-sample diagnostics.
    cfg keys: T, tau_D, rho_U, u2_max, d2_max, n_views, alpha, bits (squeeze bit depth, default 3)."""
    bits = cfg.get("bits", 3)
    preds, route, Ds, Us, n_forward = [], [], [], [], 0
    for s in range(0, len(x), bs):
        xb = x[s:s + bs]
        sig = probe(model, xb, cfg["T"], cfg.get("alpha", 0.5), bits=bits, device=device)
        n_forward += 2 * len(xb)
        pred = sig["y_fast"].clone()
        r = torch.zeros(len(xb), dtype=torch.long)                         # 0 fast, 1 stage-2, 2 abstain
        hot = (sig["D"] >= cfg["tau_D"]) | (sig["U"] >= cfg["rho_U"])
        if hot.any():
            st = stage2(model, xb[hot], cfg["n_views"], cfg["T"], bits=bits, seed=seed + s, device=device)
            n_forward += cfg["n_views"] * int(hot.sum())
            ok = (st["U2"] < cfg["u2_max"]) & (st["D2"] < cfg["d2_max"])
            y2 = torch.where(ok, st["y2"], torch.full_like(st["y2"], -1))
            pred[hot] = y2
            r[hot] = torch.where(ok, 1, 2)
        preds.append(pred); route.append(r); Ds.append(sig["D"]); Us.append(sig["U"])
    return {"pred": torch.cat(preds), "route": torch.cat(route), "D": torch.cat(Ds), "U": torch.cat(Us),
            "forward_passes_per_image": n_forward / len(x)}


def baseline_predict(model, x, method, n_views=10, device="cpu", bs=250, seed=0, bits=3):
    """Single-defence baselines under matched conditions."""
    out = []
    for s in range(0, len(x), bs):
        xb = x[s:s + bs].to(device)
        with torch.no_grad():
            if method == "none":
                out.append(model(xb).argmax(1).cpu())
            elif method == "squeeze_only":                          # Xu et al. squeezing as a preprocessor
                out.append(model(squeeze(xb, bits)).argmax(1).cpu())
            elif method == "randomization_only":                    # Xie et al. on the raw input
                g = torch.Generator().manual_seed(seed + s)
                v = torch.cat([random_resize_pad(xb, generator=g) for _ in range(n_views)])
                out.append(F.softmax(model(v), -1).view(n_views, len(xb), -1).mean(0).argmax(1).cpu())
            elif method == "always_stage2":                          # squeeze + randomisation on every input
                out.append(stage2(model, xb, n_views, bits=bits, seed=seed + s, device=device)["y2"])
    return torch.cat(out)


# =============================================================================================
# Offline calibration (blocks C-D): temperature, alpha, (tau_D, rho_U), Stage-2 acceptance
# =============================================================================================
def fit_temperature(logits, labels):
    """Minimise validation NLL over a single scalar T > 0 (Guo et al., 2017)."""
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())


def ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    bins = torch.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.float().mean() * ((pred[m] == labels[m]).float().mean() - conf[m].mean()).abs()
    return float(e)


def auroc(scores_clean, scores_attack):
    """Probability that a random attacked sample scores higher than a random clean one (Mann-Whitney)."""
    s = np.concatenate([scores_clean, scores_attack])
    lab = np.concatenate([np.zeros(len(scores_clean)), np.ones(len(scores_attack))])
    ranks = s.argsort().argsort() + 1.0
    n1, n0 = lab.sum(), len(lab) - lab.sum()
    return float((ranks[lab == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def select_thresholds(D_clean, U_clean, D_att, U_att, max_clean_route=0.10, grid=101):
    """Choose (tau_D, rho_U) on VALIDATION data: maximise the fraction of attacked inputs routed to
    Stage 2 subject to routing at most `max_clean_route` of clean inputs (a declared operating point)."""
    best = None
    qs = np.linspace(0.0, 1.0, grid)
    tD = np.unique(np.quantile(D_clean, qs))
    tU = np.unique(np.quantile(U_clean, qs))
    for td in tD:
        cd = D_clean >= td
        ad = D_att >= td
        for tu in tU:
            fpr = float(np.mean(cd | (U_clean >= tu)))
            if fpr > max_clean_route:
                continue
            tpr = float(np.mean(ad | (U_att >= tu)))
            if best is None or tpr > best["tpr"] or (tpr == best["tpr"] and fpr < best["fpr"]):
                best = {"tau_D": float(td), "rho_U": float(tu), "tpr": tpr, "fpr": fpr}
    return best


def select_alpha(D_clean, U_clean, D_att, U_att):
    """alpha for the scalar summary TS = alpha*D + (1-alpha)*U, chosen by validation AUROC."""
    best = (0.5, -1)
    for a in np.linspace(0, 1, 21):
        s = auroc(a * D_clean + (1 - a) * U_clean, a * D_att + (1 - a) * U_att)
        if s > best[1]:
            best = (float(a), s)
    return best


# =============================================================================================
# Attacks (all in pixel space [0,1])
# =============================================================================================
def fgsm(model, x, y, eps):
    x = x.clone().requires_grad_(True)
    g, = torch.autograd.grad(F.cross_entropy(model(x), y), x)
    return (x + eps * g.sign()).clamp(0, 1).detach()


def bim(model, x, y, eps, steps=20, step_size=None):
    """Basic Iterative Method / I-FGSM (Kurakin et al., 2017): no random start, L_inf projection."""
    step_size = step_size or eps / 8
    xa = x.clone()
    for _ in range(steps):
        xa.requires_grad_(True)
        g, = torch.autograd.grad(F.cross_entropy(model(xa), y), xa)
        xa = (xa + step_size * g.sign()).detach()
        xa = torch.min(torch.max(xa, x - eps), x + eps).clamp(0, 1)
    return xa


def cw_l2(model, x, y, steps=100, c_init=1.0, search=4, kappa=0.0, lr=0.01):
    """Untargeted Carlini-Wagner L2 (2017) with tanh change of variables and binary search on c.
    Returns the successful adversarial example with the smallest L2 norm (original x if none)."""
    n, dev = x.shape[0], x.device
    lo, hi = torch.zeros(n, device=dev), torch.full((n,), 1e10, device=dev)
    c = torch.full((n,), c_init, device=dev)
    best_l2 = torch.full((n,), 1e10, device=dev)
    best_adv = x.clone()
    onehot = F.one_hot(y, K).bool()
    w0 = torch.atanh((x * 2 - 1).clamp(-0.999999, 0.999999))
    for _ in range(search):
        w = w0.clone().requires_grad_(True)
        opt = torch.optim.Adam([w], lr=lr)
        succeeded = torch.zeros(n, dtype=torch.bool, device=dev)
        for _ in range(steps):
            xa = (torch.tanh(w) + 1) / 2
            z = model(xa)
            real = z[onehot]
            other = z.masked_fill(onehot, -1e9).max(1).values
            f = (real - other + kappa).clamp_min(0)
            l2 = ((xa - x) ** 2).flatten(1).sum(1)
            loss = (l2 + c * f).sum()
            opt.zero_grad()
            loss.backward()
            opt.step()
            with torch.no_grad():
                win = (z.argmax(1) != y) & (l2 < best_l2)
                best_l2[win] = l2[win].detach()
                best_adv[win] = xa[win].detach()
                succeeded |= (z.argmax(1) != y)
        hi = torch.where(succeeded, torch.minimum(hi, c), hi)
        lo = torch.where(~succeeded, torch.maximum(lo, c), lo)
        c = torch.where(hi < 1e9, (lo + hi) / 2, c * 10)
    return best_adv.detach()


def adaptive_bpda_eot(model, x, y, eps=8 / 255, steps=20, eot=4, beta=1.0, step_size=None, seed=0, bits=3):
    """White-box adaptive attack against the COMPLETE LCD pipeline (Athalye et al. 2018; Tramer et al. 2020):
       - BPDA through the non-differentiable squeeze S,
       - EOT over Stage-2 random resize/pad views,
       - a gate-evasion term that keeps D (JS between x and S(x) predictions) small, so the
         input is pushed towards the fast path while fooling both the fast path and Stage 2."""
    step_size = step_size or eps / 8
    g = torch.Generator().manual_seed(seed)
    xa = x.clone()
    for _ in range(steps):
        xa.requires_grad_(True)
        z0 = model(xa)
        xs = squeeze_bpda(xa, bits)
        zs = model(xs)
        views = torch.cat([random_resize_pad(xs, generator=g) for _ in range(eot)])
        p_bar = F.softmax(model(views), -1).view(eot, len(x), -1).mean(0)
        # all three terms are per-image sums, so each image's own gradient balances them 1 : 1 : beta
        loss = (F.cross_entropy(z0, y, reduction="sum") + F.nll_loss(p_bar.clamp_min(1e-12).log(), y, reduction="sum")
                - beta * js_norm(F.softmax(z0, -1), F.softmax(zs, -1)).sum())
        grad, = torch.autograd.grad(loss, xa)
        xa = (xa + step_size * grad.sign()).detach()
        xa = torch.min(torch.max(xa, x - eps), x + eps).clamp(0, 1)
    return xa


def attack_in_batches(fn, model, x, y, bs=250, device=None, **kw):
    """Runs an attack batch by batch on the model's device and returns CPU tensors."""
    model.eval()
    device = device or next(model.parameters()).device
    return torch.cat([fn(model, x[s:s + bs].to(device), y[s:s + bs].to(device), **kw).cpu()
                      for s in range(0, len(x), bs)])


# =============================================================================================
# BadNets (Gu et al., 2017) and the activation-clustering audit (Chen et al., 2019)
# =============================================================================================
def add_trigger(x, size=3):
    """3x3 white square in the bottom-right corner (MTP-1 trigger)."""
    x = x.clone()
    x[:, :, -size:, -size:] = 1.0
    return x


def make_poison_fn(rate=0.05, target=0, size=3, seed=0):
    g = torch.Generator().manual_seed(seed)

    def poison(xb, yb):
        m = torch.rand(len(xb), generator=g) < rate
        if m.any():
            xb = xb.clone(); yb = yb.clone()
            xb[m] = add_trigger(xb[m], size)
            yb[m] = target
        return xb, yb
    return poison


def poisoned_dataset(x, y, rate=0.05, target=0, size=3, seed=0):
    """Fixed poisoned training set (the indices are known, so the audit can be scored)."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(x), generator=g)[: int(rate * len(x))]
    xp, yp = x.clone(), y.clone()
    xp[idx] = add_trigger(xp[idx], size)
    yp[idx] = target
    flag = torch.zeros(len(x), dtype=torch.bool)
    flag[idx] = True
    return xp, yp, flag


@torch.no_grad()
def activation_clustering(model, x, y, n_components=10, device="cpu"):
    """Per class: penultimate activations -> PCA(10) -> 2-means. A class whose smaller cluster is
    unusually large and well separated is suspicious; samples in that cluster are flagged."""
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    acts = torch.cat([penultimate(model, xb.to(device)).cpu() for xb, _ in batches(x, y, 500)]).numpy()
    report, flagged = [], np.zeros(len(x), bool)
    for c in range(K):
        idx = np.where(y.numpy() == c)[0]
        a = PCA(n_components=n_components, random_state=0).fit_transform(acts[idx])
        km = KMeans(2, n_init=10, random_state=0).fit(a)
        small = np.argmin(np.bincount(km.labels_))
        frac = float(np.mean(km.labels_ == small))
        sil = float(silhouette_score(a[:3000], km.labels_[:3000]))
        report.append({"class": c, "small_cluster_fraction": frac, "silhouette": sil})
    sil_all = np.array([r["silhouette"] for r in report])
    z = (sil_all - np.median(sil_all)) / (np.median(np.abs(sil_all - np.median(sil_all))) * 1.4826 + 1e-9)
    suspect = int(np.argmax(z))
    idx = np.where(y.numpy() == suspect)[0]
    a = PCA(n_components=n_components, random_state=0).fit_transform(acts[idx])
    km = KMeans(2, n_init=10, random_state=0).fit(a)
    small = np.argmin(np.bincount(km.labels_))
    flagged[idx[km.labels_ == small]] = True
    for r, zz in zip(report, z):
        r["robust_z"] = float(zz)
    return {"suspect_class": suspect, "per_class": report, "flagged": flagged}

In [ ]:
%%writefile {ROOT}/08_Publication/train_job.py
"""Trains one model for the publication experiments and saves it atomically.

    python train_job.py --dataset cifar10 --seed 1                 # clean model, training seed 1
    python train_job.py --dataset cifar10 --poison 0.10            # BadNets model (10% poison, target class 0)
    python train_job.py --dataset svhn --seed 0 --epochs 6         # SVHN (no horizontal flips: digits)

The data split is fixed (seed 2073) for every model, so seeds vary initialisation, batch order and augmentation only.
"""
import argparse
import json
import sys
import time
from pathlib import Path

import torch

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "04_Practical_Implementation"))
import lcd_core as L  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--dataset", default="cifar10")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--poison", type=float, default=0.0)
ap.add_argument("--epochs", type=int, default=10)
ap.add_argument("--arch", default="resnet20")
ap.add_argument("--threads", type=int, default=4)
ap.add_argument("--data", required=True)
ap.add_argument("--device", default="cpu")
a = ap.parse_args()
torch.set_num_threads(a.threads)

CK = HERE / "checkpoints"
LOGS = HERE / "logs"
CK.mkdir(exist_ok=True); LOGS.mkdir(exist_ok=True)
tag = f"{a.dataset}_{a.arch}_" + (f"badnets_p{int(round(a.poison * 100))}" if a.poison else f"clean_s{a.seed}")
out = CK / f"{tag}.pt"
if out.exists():
    print("exists:", out.name); sys.exit(0)
log_f = open(LOGS / f"train_{tag}.txt", "w", encoding="utf-8")
log = lambda s: (log_f.write(s + "\n"), log_f.flush(), print(s, flush=True))

(xtr, ytr), (xva, yva), _ = L.load_dataset(a.dataset, a.data)
if a.poison:
    xtr, ytr, _ = L.poisoned_dataset(xtr, ytr, rate=a.poison, target=0, seed=3)     # same recipe as the pilot
torch.manual_seed(a.seed)
model = L.make_model(a.arch).to(a.device)
t0 = time.time()
log(f"training {tag}: {len(xtr)} images, {a.epochs} epochs, flip={a.dataset != 'svhn'}")
L.train_model(model, xtr, ytr, a.epochs, device=a.device, log=log, seed=a.seed if not a.poison else 2073, flip=(a.dataset != "svhn"))
val_acc = L.accuracy(model, xva, yva, device=a.device)
tmp = out.with_suffix(".tmp")
torch.save(model.state_dict(), tmp)
tmp.replace(out)
meta = {"tag": tag, "dataset": a.dataset, "seed": a.seed, "poison": a.poison, "epochs": a.epochs,
        "val_acc": val_acc, "train_minutes": (time.time() - t0) / 60}
(CK / f"{tag}.json").write_text(json.dumps(meta, indent=2))
log(f"done {tag}: val acc {val_acc:.4f} in {meta['train_minutes']:.1f} min")

In [ ]:
%%writefile {ROOT}/08_Publication/pub_eval.py
"""Publication-grade evaluation of Threat-Aware LCD for ONE (dataset, model seed).

    python pub_eval.py --dataset cifar10 --seed 0 --data <root>          # CPU pilot (ResNet-20)
    python pub_eval.py --dataset cifar10 --seed 0 --arch resnet18 --device cuda --data <root> --full   # Colab

Protocol (validation only for every choice; the test split is used once):
  calibration  T, then squeeze bit depth b in {3,4,5,6}, tau_D, rho_U, alpha, Stage-2 limits (block-D objective)
  attacks      FGSM eps 4/8/16, BIM 8, C&W-L2, AutoAttack (APGD-CE + APGD-DLR) transferred from the undefended model,
               AutoAttack against a BPDA+EOT surrogate of the whole pipeline (adaptive), and BPDA+EOT PGD with a
               gate-evasion term (adaptive)
  systems      none, squeeze only, randomisation only, always Stage 2, Threat-Aware LCD, and an MTP-1-style
               confidence gate (routes on 1 - max p at the same 10% clean budget)
  statistics   per-condition bootstrap 95% CIs over test images; seeds are aggregated by aggregate.py
Writes results/<dataset>_<arch>_s<seed>.json.
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "04_Practical_Implementation"))
import lcd_core as L  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--dataset", default="cifar10")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--arch", default="resnet20")
ap.add_argument("--data", required=True)
ap.add_argument("--device", default="cpu")
ap.add_argument("--threads", type=int, default=6)
ap.add_argument("--full", action="store_true", help="larger test sets and default AutoAttack iterations (GPU)")
ap.add_argument("--latency", action="store_true", help="measure batch-1 latency (only on an otherwise idle machine)")
ap.add_argument("--smoke", action="store_true", help="tiny sizes: checks every code path in a few minutes")
a = ap.parse_args()
torch.set_num_threads(a.threads)
DEV = a.device
N_TEST, N_CW, N_AA, N_ADA = (5000, 1000, 1000, 1000) if a.full else (2000, 300, 500, 500)
AA_ITERS, CW_STEPS = (100, 200) if a.full else (40, 100)
N_VAL, N_VIEWS, BUDGET = 1000, 10, 0.10
if a.smoke:
    N_TEST, N_CW, N_AA, N_ADA, AA_ITERS, CW_STEPS, N_VAL = 60, 20, 20, 20, 3, 5, 100

OUT = HERE / "results"; OUT.mkdir(exist_ok=True)
tag = f"{a.dataset}_{a.arch}_s{a.seed}" + ("_smoke" if a.smoke else "")
log_f = open(HERE / "logs" / f"eval_{tag}.txt", "w", encoding="utf-8")


def log(*x):
    s = " ".join(str(v) for v in x); print(s, flush=True); log_f.write(s + "\n"); log_f.flush()


def boot_ci(v, n_boot=2000, seed=0):
    v = np.asarray(v, float)
    if len(v) == 0:
        return [float("nan")] * 2
    rng = np.random.default_rng(seed)
    m = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(1)
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]


def rate(mask):
    m = np.asarray(mask, float)
    return {"mean": float(m.mean()), "ci95": boot_ci(m)}


t_start = time.time()
torch.manual_seed(2073 + a.seed)
np.random.seed(2073 + a.seed)
(xtr, ytr), (xva, yva), (xte, yte) = L.load_dataset(a.dataset, a.data)
ck = HERE / "checkpoints" / f"{a.dataset}_{a.arch}_clean_s{a.seed}.pt"
model = L.make_model(a.arch); model.load_state_dict(torch.load(ck, map_location=DEV)); model.to(DEV).eval()
for p_ in model.parameters():
    p_.requires_grad_(False)
R = {"dataset": a.dataset, "arch": a.arch, "seed": a.seed, "full": a.full,
     "sizes": {"test": N_TEST, "cw": N_CW, "autoattack": N_AA, "adaptive": N_ADA, "val_attack": N_VAL}}
R["clean_val_acc"] = L.accuracy(model, xva, yva, device=DEV)
log(f"[{tag}] clean val acc {R['clean_val_acc']:.4f}")

# ================================================================ calibration (validation only)
vl = L.predict_logits(model, xva, device=DEV)
T = L.fit_temperature(vl, yva)
R["calibration"] = {"T": T, "ece_before": L.ece(torch.softmax(vl, 1), yva), "ece_after": L.ece(torch.softmax(vl / T, 1), yva)}
xv, yv = xva[:N_VAL], yva[:N_VAL]
val_adv = [L.attack_in_batches(L.fgsm, model, xv, yv, eps=8 / 255),
           L.attack_in_batches(L.bim, model, xv, yv, eps=8 / 255, steps=20, step_size=1 / 255)]
study = {}
for b in (3, 4, 5, 6):
    sc = L.probe(model, xv, T, bits=b, device=DEV)
    sa = [L.probe(model, z, T, bits=b, device=DEV) for z in val_adv]
    Dc, Uc = sc["D"].numpy(), sc["U"].numpy()
    Da = np.concatenate([s["D"].numpy() for s in sa]); Ua = np.concatenate([s["U"].numpy() for s in sa])
    thr = L.select_thresholds(Dc, Uc, Da, Ua, max_clean_route=BUDGET)
    study[b] = {"AUROC_D": L.auroc(Dc, Da), "AUROC_U": L.auroc(Uc, Ua), "tpr": thr["tpr"], "fpr": thr["fpr"], "thr": thr,
                "Dc": Dc, "Uc": Uc, "Da": Da, "Ua": Ua}
BITS = max(study, key=lambda b: study[b]["tpr"])
s_ = study[BITS]
alpha, auc_ts = L.select_alpha(s_["Dc"], s_["Uc"], s_["Da"], s_["Ua"])
st = L.stage2(model, xva[:200 if a.smoke else 2000], N_VIEWS, T, bits=BITS, seed=11, device=DEV)
cfg = {"T": T, "bits": BITS, "tau_D": s_["thr"]["tau_D"], "rho_U": s_["thr"]["rho_U"], "alpha": alpha, "n_views": N_VIEWS,
       "u2_max": float(np.quantile(st["U2"].numpy(), 0.95)), "d2_max": float(np.quantile(st["D2"].numpy(), 0.95))}
# MTP-1-style confidence gate at the same clean budget: route if 1 - max p0 >= its 90th clean percentile
conf_clean = 1 - L.probe(model, xv, T, bits=BITS, device=DEV)["p0"].max(1).values.numpy()
conf_att = np.concatenate([1 - L.probe(model, z, T, bits=BITS, device=DEV)["p0"].max(1).values.numpy() for z in val_adv])
conf_thr = float(np.quantile(conf_clean, 1 - BUDGET))
R["validation"] = {"bit_study": {b: {k: v for k, v in study[b].items() if k in ("AUROC_D", "AUROC_U", "tpr", "fpr")} for b in study},
                   "selected_bits": BITS, "AUROC_TS": auc_ts, "alpha": alpha,
                   "AUROC_maxprob": L.auroc(conf_clean, conf_att), "conf_gate_tpr": float(np.mean(conf_att >= conf_thr))}
R["frozen_config"] = {**cfg, "conf_thr": conf_thr}
log(f"[{tag}] bits={BITS} tau_D={cfg['tau_D']:.4f} rho_U={cfg['rho_U']:.4f} alpha={alpha:.2f} | val AUROC D "
    f"{s_['AUROC_D']:.3f} maxprob {R['validation']['AUROC_maxprob']:.3f} | val TPR LCD {s_['tpr']:.3f} conf {R['validation']['conf_gate_tpr']:.3f}")


# ================================================================ defended predictors
def conf_gate_predict(x, seed=7, bs=250):
    """Same Stage 2 and abstention as LCD, but the gate is the MTP-1 confidence score."""
    preds, routes = [], []
    for s in range(0, len(x), bs):
        xb = x[s:s + bs]
        sig = L.probe(model, xb, T, bits=BITS, device=DEV)
        pred = sig["y_fast"].clone(); r = torch.zeros(len(xb), dtype=torch.long)
        hot = (1 - sig["p0"].max(1).values) >= conf_thr
        if hot.any():
            st2 = L.stage2(model, xb[hot], N_VIEWS, T, bits=BITS, seed=seed + s, device=DEV)
            ok = (st2["U2"] < cfg["u2_max"]) & (st2["D2"] < cfg["d2_max"])
            pred[hot] = torch.where(ok, st2["y2"], torch.full_like(st2["y2"], -1)); r[hot] = torch.where(ok, 1, 2)
        preds.append(pred); routes.append(r)
    return torch.cat(preds), torch.cat(routes)


def evaluate(x, y):
    out = {}
    for sname in ("none", "squeeze_only", "randomization_only", "always_stage2"):
        pr = L.baseline_predict(model, x, sname, N_VIEWS, device=DEV, seed=5, bits=BITS)
        out[sname] = {"acc": rate((pr == y).numpy())}
    o = L.lcd_predict(model, x, cfg, device=DEV, seed=7)
    cov = o["pred"] != -1
    out["LCD"] = {"acc": rate((o["pred"] == y).numpy()), "coverage": rate(cov.numpy()),
                  "wrong_accepted": rate(((o["pred"] != y) & cov).numpy()),
                  "routed": rate((o["route"] > 0).numpy()), "abstain": rate((o["route"] == 2).numpy()),
                  "passes": o["forward_passes_per_image"]}
    pc, rc = conf_gate_predict(x)
    covc = pc != -1
    out["conf_gate"] = {"acc": rate((pc == y).numpy()), "coverage": rate(covc.numpy()),
                        "wrong_accepted": rate(((pc != y) & covc).numpy()), "routed": rate((rc > 0).numpy())}
    return out


# ================================================================ attacks on the test split
xt, yt = xte[:N_TEST], yte[:N_TEST]
conds = {"clean": xt}
for e in (4, 8, 16):
    conds[f"FGSM_{e}"] = L.attack_in_batches(L.fgsm, model, xt, yt, eps=e / 255, device=DEV)
conds["BIM_8"] = L.attack_in_batches(L.bim, model, xt, yt, eps=8 / 255, steps=20, step_size=1 / 255, device=DEV)
log(f"[{tag}] FGSM/BIM done ({(time.time() - t_start) / 60:.0f} min)")
conds["CW_L2"] = L.attack_in_batches(L.cw_l2, model, xt[:N_CW], yt[:N_CW], steps=CW_STEPS, search=3, device=DEV)
log(f"[{tag}] C&W done ({(time.time() - t_start) / 60:.0f} min)")

from autoattack import AutoAttack  # noqa: E402
import autoattack.autopgd_base as _apb  # noqa: E402


def _check_zero_gradients(grad, logger=None):
    """Same diagnostic as autoattack.checks.check_zero_gradients, but with reshape(): gradients that flow through the
    BPDA squeeze / resize views are not contiguous, and the original .view() call raises on them."""
    z = grad.reshape(grad.shape[0], -1).abs().sum(-1)
    if (z == 0).any():
        msg = f"there are {int((z == 0).sum())} points with zero gradient"
        print("warning:", msg) if logger is None else logger.log(msg)


_apb.check_zero_gradients = _check_zero_gradients


def run_aa(net, x, y, eot_note):
    adv = AutoAttack(net, norm="Linf", eps=8 / 255, version="custom", attacks_to_run=["apgd-ce", "apgd-dlr"], device=DEV,
                     verbose=False)
    adv.apgd.n_iter = AA_ITERS
    adv.apgd.n_restarts = 1
    adv.apgd_targeted.n_iter = AA_ITERS
    out = []
    for s in range(0, len(x), 250):
        out.append(adv.run_standard_evaluation(x[s:s + 250].to(DEV), y[s:s + 250].to(DEV), bs=250).cpu())
    return torch.cat(out)


conds["AutoAttack_transfer"] = run_aa(model, xt[:N_AA], yt[:N_AA], "none")
log(f"[{tag}] AutoAttack (transfer) done ({(time.time() - t_start) / 60:.0f} min)")


class PipelineSurrogate(nn.Module):
    """Differentiable surrogate of the whole defence for an adaptive AutoAttack: BPDA through the squeeze and EOT over
    Stage-2 views, mixing the fast-path and Stage-2 distributions so both routes are attacked."""

    def __init__(self, eot=4):
        super().__init__(); self.eot = eot

    def forward(self, x):
        if x.requires_grad:                             # gradients through resize/median come back non-contiguous;
            x.register_hook(lambda g: g.contiguous())   # AutoAttack's gradient check needs a contiguous tensor
        xs = L.squeeze_bpda(x, BITS)
        views = torch.cat([L.random_resize_pad(xs) for _ in range(self.eot)])
        p_bar = F.softmax(model(views) / T, -1).view(self.eot, len(x), -1).mean(0)
        p0 = F.softmax(model(x) / T, -1)
        return torch.log(0.5 * p0 + 0.5 * p_bar + 1e-12)


conds["AutoAttack_adaptive"] = run_aa(PipelineSurrogate().to(DEV).eval(), xt[:N_AA], yt[:N_AA], "bpda+eot")
log(f"[{tag}] AutoAttack (adaptive) done ({(time.time() - t_start) / 60:.0f} min)")
conds["BPDA_EOT_PGD"] = L.attack_in_batches(L.adaptive_bpda_eot, model, xt[:N_ADA], yt[:N_ADA], eps=8 / 255, steps=20, eot=4,
                                            beta=1.0, bits=BITS, device=DEV)
log(f"[{tag}] BPDA+EOT PGD done ({(time.time() - t_start) / 60:.0f} min)")

# ================================================================ evaluation
R["test"] = {}
for cname, xc in conds.items():
    yc = yt[: len(xc)]
    R["test"][cname] = evaluate(xc, yc)
    t = R["test"][cname]
    log(f"   {cname:20s} none {t['none']['acc']['mean']:.3f} | sq {t['squeeze_only']['acc']['mean']:.3f} | rnd "
        f"{t['randomization_only']['acc']['mean']:.3f} | S2 {t['always_stage2']['acc']['mean']:.3f} | LCD "
        f"{t['LCD']['acc']['mean']:.3f} (routed {t['LCD']['routed']['mean']:.3f}) | conf-gate {t['conf_gate']['acc']['mean']:.3f}")

# gate quality on test: AUROC of TS and of max-prob, routing of successful attacks
cs = L.probe(model, xt, T, bits=BITS, device=DEV)
ts_clean = alpha * cs["D"].numpy() + (1 - alpha) * cs["U"].numpy()
mp_clean = 1 - cs["p0"].max(1).values.numpy()
R["gate"] = {"clean_route_rate": float(((cs["D"] >= cfg["tau_D"]) | (cs["U"] >= cfg["rho_U"])).float().mean())}
for cname in [c for c in conds if c != "clean"]:
    s = L.probe(model, conds[cname], T, bits=BITS, device=DEV); yc = yt[: len(conds[cname])]
    fooled = (s["y_fast"] != yc).numpy()
    routed = ((s["D"] >= cfg["tau_D"]) | (s["U"] >= cfg["rho_U"])).numpy()
    R["gate"][cname] = {"AUROC_TS": L.auroc(ts_clean, alpha * s["D"].numpy() + (1 - alpha) * s["U"].numpy()),
                        "AUROC_maxprob": L.auroc(mp_clean, 1 - s["p0"].max(1).values.numpy()),
                        "routed_successful": float(routed[fooled].mean()) if fooled.any() else None,
                        "base_attack_success": float(fooled.mean())}

# ablations on clean / FGSM-8 / BIM-8
variants = {"full": cfg, "D_only": {**cfg, "rho_U": 1.01}, "U_only": {**cfg, "tau_D": 1.01},
            "no_abstention": {**cfg, "u2_max": 9, "d2_max": 9}, "N1": {**cfg, "n_views": 1}, "N5": {**cfg, "n_views": 5}}
R["ablations"] = {}
for v, vc in variants.items():
    R["ablations"][v] = {}
    for c in ("clean", "FGSM_8", "BIM_8"):
        o = L.lcd_predict(model, conds[c], vc, device=DEV, seed=7)
        R["ablations"][v][c] = {"acc": float((o["pred"] == yt).float().mean()), "coverage": float((o["pred"] != -1).float().mean()),
                                "passes": o["forward_passes_per_image"]}

if a.latency:
    def ms(fn, xs):
        for i in range(5): fn(xs[i:i + 1])
        if DEV == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for i in range(len(xs)): fn(xs[i:i + 1])
        if DEV == "cuda": torch.cuda.synchronize()
        return (time.perf_counter() - t0) / len(xs) * 1000
    xl = xt[:200]
    lat = {"none": ms(lambda z: L.baseline_predict(model, z, "none", device=DEV), xl),
           "LCD": ms(lambda z: L.lcd_predict(model, z, cfg, device=DEV), xl),
           "always_stage2": ms(lambda z: L.baseline_predict(model, z, "always_stage2", N_VIEWS, device=DEV, bits=BITS), xl)}
    lat.update(LCD_x=lat["LCD"] / lat["none"], always_stage2_x=lat["always_stage2"] / lat["none"])
    R["latency_ms"] = lat

R["minutes"] = (time.time() - t_start) / 60
(OUT / f"{tag}.json").write_text(json.dumps(R, indent=2), encoding="utf-8")
log(f"[{tag}] done in {R['minutes']:.0f} min")

In [ ]:
%%writefile {ROOT}/08_Publication/pub_backdoor.py
"""Backdoor track for the paper: BadNets at 1%, 5%, 10% poison (CIFAR-10) plus a clean model as negative control.

For every model: clean accuracy; trigger attack-success rate (ASR) without defence, with squeezing, with always-on Stage 2
and with Threat-Aware LCD (frozen configuration of the clean model, temperature refitted); gate routing of triggered
inputs; and two offline audits -- activation clustering [Chen et al. 2019] and Neural Cleanse [Wang et al. 2019].

    python pub_backdoor.py --data <cifar root> [--device cuda] [--arch resnet20]
Writes results/backdoor_<arch>.json. Needs results/cifar10_<arch>_s0.json (frozen LCD configuration).
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "04_Practical_Implementation"))
import lcd_core as L  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True)
ap.add_argument("--arch", default="resnet20")
ap.add_argument("--device", default="cpu")
ap.add_argument("--threads", type=int, default=6)
ap.add_argument("--nc_steps", type=int, default=150)
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args()
torch.set_num_threads(a.threads)
DEV = a.device
N_TEST = 200 if a.smoke else 2000
NC_STEPS = 3 if a.smoke else a.nc_steps
log_f = open(HERE / "logs" / f"backdoor_{a.arch}{'_smoke' if a.smoke else ''}.txt", "w", encoding="utf-8")


def log(*x):
    s = " ".join(str(v) for v in x); print(s, flush=True); log_f.write(s + "\n"); log_f.flush()


def neural_cleanse(model, x_clean, steps, n_classes=10, lam=1e-2, lr=0.1, bs=128, seed=0):
    """Wang et al. (S&P 2019): for each class, find the smallest mask+pattern that sends clean images to that class.
    A backdoor target needs an abnormally small mask; anomaly index = (median - L1) / (1.4826 * MAD)."""
    g = torch.Generator().manual_seed(seed)
    norms, success = [], []
    for t in range(n_classes):
        m_raw = torch.full((1, 1, 32, 32), -3.0, device=DEV, requires_grad=True)
        p_raw = torch.zeros((1, 3, 32, 32), device=DEV, requires_grad=True)
        opt = torch.optim.Adam([m_raw, p_raw], lr=lr)
        tgt = torch.full((bs,), t, dtype=torch.long, device=DEV)
        for _ in range(steps):
            idx = torch.randint(0, len(x_clean), (bs,), generator=g)
            xb = x_clean[idx].to(DEV)
            m, p = torch.sigmoid(m_raw), torch.sigmoid(p_raw)
            loss = F.cross_entropy(model((1 - m) * xb + m * p), tgt) + lam * m.sum()
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            m, p = torch.sigmoid(m_raw), torch.sigmoid(p_raw)
            xb = x_clean[:500].to(DEV)
            success.append(float((model((1 - m) * xb + m * p).argmax(1) == t).float().mean()))
            norms.append(float(m.sum()))
    norms = np.array(norms)
    med = np.median(norms); mad = np.median(np.abs(norms - med)) * 1.4826 + 1e-9
    anomaly = (med - norms) / mad                       # large positive = abnormally small trigger
    flagged = int(np.argmax(anomaly))
    return {"mask_l1": norms.round(2).tolist(), "reversed_trigger_success": success, "anomaly_index": anomaly.round(3).tolist(),
            "flagged_class": flagged, "max_anomaly": float(anomaly.max()), "detected": bool(anomaly.max() > 2.0)}


t0 = time.time()
(xtr, ytr), (xva, yva), (xte, yte) = L.load_dataset("cifar10", a.data)
cfg0 = json.loads((HERE / "results" / f"cifar10_{a.arch}_s0.json").read_text(encoding="utf-8"))["frozen_config"]
BITS = cfg0["bits"]
xt, yt = xte[:N_TEST], yte[:N_TEST]
nt = yt != 0
xtrig = L.add_trigger(xt[nt])
R = {"arch": a.arch, "n_test": N_TEST, "nc_steps": NC_STEPS, "bits": BITS, "models": {}}
models = [("clean_control", f"cifar10_{a.arch}_clean_s0.pt", 0.0)] + \
         [(f"badnets_{p}pct", f"cifar10_{a.arch}_badnets_p{p}.pt", p / 100) for p in (1, 5, 10)]
for name, fname, rate in models:
    ck = HERE / "checkpoints" / fname
    if not ck.exists():
        log(f"skip {name}: {fname} not trained yet"); continue
    m = L.make_model(a.arch); m.load_state_dict(torch.load(ck, map_location=DEV)); m.to(DEV).eval()
    for q in m.parameters():
        q.requires_grad_(False)
    Tm = L.fit_temperature(L.predict_logits(m, xva, device=DEV), yva)
    cfg = {**{k: cfg0[k] for k in ("tau_D", "rho_U", "u2_max", "d2_max", "n_views", "bits", "alpha")}, "T": Tm}
    o = L.lcd_predict(m, xtrig, cfg, device=DEV, seed=9)
    cov = o["pred"] != -1
    r = {"poison_rate": rate,
         "clean_acc": L.accuracy(m, xt, yt, device=DEV),
         "ASR_none": float((L.predict_logits(m, xtrig, device=DEV).argmax(1) == 0).float().mean()),
         "ASR_squeeze": float((L.baseline_predict(m, xtrig, "squeeze_only", device=DEV, bits=BITS) == 0).float().mean()),
         "ASR_always_stage2": float((L.baseline_predict(m, xtrig, "always_stage2", cfg["n_views"], device=DEV, bits=BITS) == 0).float().mean()),
         "ASR_LCD": float(((o["pred"] == 0) & cov).float().mean()),
         "LCD_routed_triggered": float((o["route"] > 0).float().mean())}
    if rate:
        xp, yp, flag = L.poisoned_dataset(xtr, ytr, rate=rate, target=0, seed=3)
    else:                                               # control: audit the clean training set
        xp, yp, flag = xtr, ytr, torch.zeros(len(xtr), dtype=torch.bool)
    ac = L.activation_clustering(m, xp, yp, device=DEV)
    fl, pf = ac["flagged"], flag.numpy()
    zs = [c["robust_z"] for c in ac["per_class"]]
    r["activation_clustering"] = {"suspect_class": ac["suspect_class"], "max_robust_z": float(max(zs)),
                                  "precision": float((fl & pf).sum() / max(fl.sum(), 1)) if rate else None,
                                  "recall": float((fl & pf).sum() / max(pf.sum(), 1)) if rate else None,
                                  "flagged": int(fl.sum())}
    r["neural_cleanse"] = neural_cleanse(m, xva[:2000], NC_STEPS)
    R["models"][name] = r
    log(f"{name}: clean {r['clean_acc']:.3f} | ASR none {r['ASR_none']:.3f} sq {r['ASR_squeeze']:.3f} S2 {r['ASR_always_stage2']:.3f} "
        f"LCD {r['ASR_LCD']:.3f} (routed {r['LCD_routed_triggered']:.3f}) | AC class {ac['suspect_class']} z {max(zs):.2f} | "
        f"NC class {r['neural_cleanse']['flagged_class']} anomaly {r['neural_cleanse']['max_anomaly']:.2f} "
        f"({'detected' if r['neural_cleanse']['detected'] else 'not detected'}) [{(time.time() - t0) / 60:.0f} min]")
    (HERE / "results" / f"backdoor_{a.arch}{'_smoke' if a.smoke else ''}.json").write_text(json.dumps(R, indent=2), encoding="utf-8")
log(f"done in {(time.time() - t0) / 60:.0f} min")

In [ ]:
%%writefile {ROOT}/08_Publication/aggregate.py
"""Aggregates results/*.json into the paper's tables, figures and number macros.

    python aggregate.py [--arch resnet20]

Writes paper/numbers.tex (\\res{key} macros), paper/tables/*.tex, paper/figures/*.pdf and results/summary_<arch>.json.
Multi-seed cells are mean ± std over seeds; single-run cells show the bootstrap 95% CI of the run.
"""
import argparse
import glob
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path(__file__).resolve().parent
ap = argparse.ArgumentParser()
ap.add_argument("--arch", default="resnet20")
a = ap.parse_args()
RES, PAPER = HERE / "results", HERE / "paper"
(PAPER / "tables").mkdir(parents=True, exist_ok=True); (PAPER / "figures").mkdir(parents=True, exist_ok=True)

SYS = [("none", "None"), ("squeeze_only", "Squeeze"), ("randomization_only", "Random"), ("always_stage2", "Always S2"),
       ("conf_gate", "Conf.\\ gate"), ("LCD", "LCD")]
CONDS = [("clean", "Clean"), ("FGSM_4", "FGSM 4/255"), ("FGSM_8", "FGSM 8/255"), ("FGSM_16", "FGSM 16/255"),
         ("BIM_8", "BIM 8/255"), ("CW_L2", "C\\&W-$L_2$"), ("AutoAttack_transfer", "AutoAttack (transfer)"),
         ("AutoAttack_adaptive", "AutoAttack (adaptive)"), ("BPDA_EOT_PGD", "BPDA+EOT PGD (adaptive)")]
ATT = [c for c, _ in CONDS if c != "clean"]


def load(ds):
    files = sorted(f for f in glob.glob(str(RES / f"{ds}_{a.arch}_s*.json")) if "smoke" not in f)
    return [json.loads(Path(f).read_text(encoding="utf-8")) for f in files]


runs = {"cifar10": load("cifar10"), "svhn": load("svhn")}
bd_path = RES / f"backdoor_{a.arch}.json"
bd = json.loads(bd_path.read_text(encoding="utf-8")) if bd_path.exists() else None
macros = {}


def stat(vals):
    v = np.array([x for x in vals if x is not None], float)
    return (float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0, len(v)) if len(v) else (np.nan, 0.0, 0)


def cell(rs, getter, pct=True, ci_getter=None):
    m, s, n = stat([getter(r) for r in rs])
    f = 100 if pct else 1
    if np.isnan(m):
        return "--", m
    if n > 1:
        return (f"{m * f:.1f}$\\pm${s * f:.1f}" if pct else f"{m:.3f}$\\pm${s:.3f}"), m
    if ci_getter is not None:
        lo, hi = ci_getter(rs[0])
        return (f"{m * f:.1f} [{lo * f:.0f},{hi * f:.0f}]" if pct else f"{m:.3f}"), m
    return (f"{m * f:.1f}" if pct else f"{m:.3f}"), m


def acc_cell(rs, cond, sysk):
    return cell(rs, lambda r: r["test"][cond][sysk]["acc"]["mean"], ci_getter=lambda r: r["test"][cond][sysk]["acc"]["ci95"])


def write_table(name, header, rows, colspec, caption, label, star=False, size="\\footnotesize"):
    env = "table*" if star else "table"
    lines = [f"\\begin{{{env}}}[t]", "\\centering", size, f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             "\\setlength{\\tabcolsep}{3pt}", "\\begin{adjustbox}{max width=\\linewidth}", f"\\begin{{tabular}}{{{colspec}}}",
             "\\toprule", " & ".join(header) + " \\\\", "\\midrule"] + [" & ".join(r) + " \\\\" for r in rows] + \
            ["\\bottomrule", "\\end{tabular}", "\\end{adjustbox}", f"\\end{{{env}}}"]
    (PAPER / "tables" / f"{name}.tex").write_text("\n".join(lines), encoding="utf-8")


# ======================================================================== main accuracy tables
for ds, rs in runs.items():
    if not rs:
        continue
    rows = []
    for c, cname in CONDS:
        row = [cname]
        best = max(stat([r["test"][c][k]["acc"]["mean"] for r in rs])[0] for k, _ in SYS if k != "none")
        for k, _ in SYS:
            txt, m = acc_cell(rs, c, k)
            macros[f"{ds}-{c}-{k}"] = f"{m * 100:.1f}"
            row.append(f"\\textbf{{{txt}}}" if k != "none" and abs(m - best) < 1e-9 else txt)
        cov, _ = cell(rs, lambda r: r["test"][c]["LCD"]["coverage"]["mean"])
        wr, wm = cell(rs, lambda r: r["test"][c]["LCD"]["wrong_accepted"]["mean"])
        rt, rm = cell(rs, lambda r: r["test"][c]["LCD"]["routed"]["mean"])
        macros[f"{ds}-{c}-LCD-cov"] = f"{stat([r['test'][c]['LCD']['coverage']['mean'] for r in rs])[0] * 100:.1f}"
        macros[f"{ds}-{c}-LCD-wrong"] = f"{wm * 100:.1f}"
        macros[f"{ds}-{c}-LCD-routed"] = f"{rm * 100:.1f}"
        row += [cov, wr]
        rows.append(row)
    nseeds = len(rs)
    what = f"mean$\\pm$std over {nseeds} seeds" if nseeds > 1 else "single run, bootstrap 95\\% CI in brackets"
    dsname = "CIFAR-10" if ds == "cifar10" else "SVHN"
    write_table(f"main_{ds}", ["Condition"] + [n for _, n in SYS] + ["LCD cov.", "LCD wrong-acc."], rows, "l" + "c" * (len(SYS) + 2),
                f"{dsname} held-out accuracy (\\%, abstentions counted as wrong; {what}). Best defended system in bold. "
                "\\emph{LCD cov.}\\ = share of inputs answered; \\emph{wrong-acc.}\\ = share answered incorrectly.",
                f"tab:main_{ds}", star=True)

# ======================================================================== gate quality
rs = runs["cifar10"]
if rs:
    rows = []
    for c in ATT:
        cname = dict(CONDS)[c]
        ts, tm = cell(rs, lambda r: r["gate"][c]["AUROC_TS"], pct=False)
        mp, mm = cell(rs, lambda r: r["gate"][c]["AUROC_maxprob"], pct=False)
        ro, rm = cell(rs, lambda r: r["gate"][c]["routed_successful"])
        bs_, bm = cell(rs, lambda r: r["gate"][c]["base_attack_success"])
        macros[f"auc-ts-{c}"] = f"{tm:.2f}"; macros[f"auc-mp-{c}"] = f"{mm:.2f}"; macros[f"routed-{c}"] = f"{rm * 100:.1f}"
        rows.append([cname, ts, mp, ro, bs_])
    write_table("gate", ["Attack", "AUROC $D$/TS", "AUROC $1-\\max p$", "Routed (\\%)", "Base ASR (\\%)"], rows, "lcccc",
                "Gate quality on CIFAR-10 test images. AUROC compares clean and attacked inputs; values below 0.5 mean the "
                "score ranks attacked inputs as \\emph{less} suspicious than clean ones. \\emph{Routed} is the share of successful "
                "attacks sent to Stage~2.", "tab:gate", star=True)
    vm = stat([r["validation"]["AUROC_maxprob"] for r in rs]); vd = stat([r["validation"]["bit_study"][str(r["validation"]["selected_bits"])]["AUROC_D"] for r in rs])
    macros["val-auc-mp"] = f"{vm[0]:.2f}"; macros["val-auc-d"] = f"{vd[0]:.2f}"
    macros["val-auc-mp-std"] = f"{vm[1]:.2f}"; macros["val-auc-d-std"] = f"{vd[1]:.2f}"
    macros["conf-tpr"] = f"{stat([r['validation']['conf_gate_tpr'] for r in rs])[0] * 100:.1f}"
    macros["lcd-tpr"] = f"{stat([r['validation']['bit_study'][str(r['validation']['selected_bits'])]['tpr'] for r in rs])[0] * 100:.1f}"
    macros["clean-acc"] = f"{stat([r['test']['clean']['none']['acc']['mean'] for r in rs])[0] * 100:.1f}"
    macros["nseeds"] = str(len(rs))
    macros["bits-selected"] = ", ".join(str(r["validation"]["selected_bits"]) for r in rs)
    macros["clean-route"] = f"{stat([r['gate']['clean_route_rate'] for r in rs])[0] * 100:.1f}"

    # ---------------------------------------------------------------- bit-depth study
    rows = []
    for b in ("3", "4", "5", "6"):
        d_ = cell(rs, lambda r: r["validation"]["bit_study"][b]["AUROC_D"], pct=False)[0]
        t_ = cell(rs, lambda r: r["validation"]["bit_study"][b]["tpr"])[0]
        sel = sum(int(r["validation"]["selected_bits"]) == int(b) for r in rs)
        rows.append([f"{b}-bit", d_, t_, f"{sel}/{len(rs)}"])
    write_table("bits", ["Bit depth $b$", "AUROC of $D$", "Attacks routed (\\%)", "Selected"], rows, "lccc",
                "Validation-only choice of the squeeze bit depth at a 10\\% clean-routing budget (CIFAR-10).", "tab:bits")

    # ---------------------------------------------------------------- ablations and cost
    rows = []
    names = {"full": "Full LCD", "D_only": "$D$ only", "U_only": "$U$ only", "no_abstention": "No abstention", "N1": "$N=1$", "N5": "$N=5$"}
    for v, vn in names.items():
        row = [vn]
        for c in ("clean", "FGSM_8", "BIM_8"):
            row.append(cell(rs, lambda r: r["ablations"][v][c]["acc"])[0])
        row.append(cell(rs, lambda r: r["ablations"][v]["clean"]["passes"], pct=False)[0].replace(".000", ""))
        rows.append(row)
        macros[f"abl-{v}-fgsm"] = f"{stat([r['ablations'][v]['FGSM_8']['acc'] for r in rs])[0] * 100:.1f}"
    write_table("ablation", ["Variant", "Clean", "FGSM 8/255", "BIM 8/255", "Passes (clean)"], rows, "lcccc",
                "Ablations on CIFAR-10 (accuracy \\%, abstentions counted as wrong).", "tab:ablation")
    lat = [r["latency_ms"] for r in rs if "latency_ms" in r]
    if lat:
        macros["lat-lcd"] = f"{lat[0]['LCD_x']:.2f}"; macros["lat-s2"] = f"{lat[0]['always_stage2_x']:.2f}"
    macros["passes-clean"] = f"{stat([r['test']['clean']['LCD']['passes'] for r in rs])[0]:.2f}"
    macros["passes-fgsm"] = f"{stat([r['test']['FGSM_8']['LCD']['passes'] for r in rs])[0]:.2f}"

# ======================================================================== backdoor table
if bd:
    rows = []
    for name, r in bd["models"].items():
        label = "Clean (control)" if name == "clean_control" else f"BadNets {int(round(r['poison_rate'] * 100))}\\%"
        acl = r["activation_clustering"]; nc = r["neural_cleanse"]
        ac_txt = (f"{acl['suspect_class']} ({acl['precision'] * 100:.0f}/{acl['recall'] * 100:.0f})" if acl["precision"] is not None
                  else f"{acl['suspect_class']} (z={acl['max_robust_z']:.1f})")
        nc_txt = f"{nc['flagged_class']} ({nc['max_anomaly']:.1f}){'*' if nc['detected'] else ''}"
        rows.append([label, f"{r['clean_acc'] * 100:.1f}", f"{r['ASR_none'] * 100:.1f}", f"{r['ASR_always_stage2'] * 100:.1f}",
                     f"{r['ASR_LCD'] * 100:.1f}", f"{r['LCD_routed_triggered'] * 100:.1f}", ac_txt, nc_txt])
        key = name.replace("_", "")
        macros[f"bd-{key}-asr"] = f"{r['ASR_none'] * 100:.1f}"; macros[f"bd-{key}-lcd"] = f"{r['ASR_LCD'] * 100:.1f}"
        macros[f"bd-{key}-routed"] = f"{r['LCD_routed_triggered'] * 100:.1f}"; macros[f"bd-{key}-s2"] = f"{r['ASR_always_stage2'] * 100:.1f}"
        if acl["precision"] is not None:
            macros[f"bd-{key}-acp"] = f"{acl['precision'] * 100:.1f}"; macros[f"bd-{key}-acr"] = f"{acl['recall'] * 100:.1f}"
        macros[f"bd-{key}-nc"] = f"{nc['max_anomaly']:.1f}"; macros[f"bd-{key}-ncclass"] = str(nc["flagged_class"])
        macros[f"bd-{key}-acclass"] = str(acl["suspect_class"])
    write_table("backdoor", ["Model", "Clean acc.", "ASR none", "ASR always S2", "ASR LCD", "Routed", "AC class (P/R)", "NC class (idx)"],
                rows, "lccccccc",
                "Backdoor track (CIFAR-10, target class 0). ASR = triggered non-target images classified as the target (\\%). "
                "AC = activation clustering: flagged class (precision/recall of flagged training samples, \\%). NC = Neural Cleanse: "
                "flagged class (anomaly index; * = index $>2$, i.e.\\ detected).", "tab:backdoor", star=True)

# ======================================================================== figures
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]


def style(ax):
    ax.spines[["top", "right"]].set_visible(False); ax.tick_params(colors=MUTED, labelsize=8)
    ax.grid(True, axis="y", color=GRID, lw=0.7); ax.set_axisbelow(True)


rs = runs["cifar10"]
if rs:
    short = {"clean": "Clean", "FGSM_4": "FGSM\n4", "FGSM_8": "FGSM\n8", "FGSM_16": "FGSM\n16", "BIM_8": "BIM\n8", "CW_L2": "C&W",
             "AutoAttack_transfer": "AA\ntrans.", "AutoAttack_adaptive": "AA\nadapt.", "BPDA_EOT_PGD": "BPDA\n+EOT"}
    fig, ax = plt.subplots(figsize=(7.2, 2.6))
    xk = np.arange(len(CONDS)); w = 0.8 / len(SYS)
    for i, (k, nm) in enumerate(SYS):
        ms = [stat([r["test"][c][k]["acc"]["mean"] for r in rs]) for c, _ in CONDS]
        ax.bar(xk + (i - (len(SYS) - 1) / 2) * w, [m[0] for m in ms], w * 0.9, yerr=[m[1] for m in ms], color=SLOTS[i],
               label=nm.replace("\\ ", " ").replace("\\", ""), error_kw=dict(lw=0.6, capsize=1.2, ecolor=INK))
    ax.set_xticks(xk, [short[c] for c, _ in CONDS], fontsize=7.5, color=INK)
    ax.set_ylabel("Accuracy", color=INK, fontsize=8); ax.set_ylim(0, 1)
    style(ax); ax.legend(frameon=False, ncol=6, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, 1.17))
    fig.tight_layout(); fig.savefig(PAPER / "figures" / "accuracy.pdf"); fig.savefig(PAPER / "figures" / "accuracy.png", dpi=200); plt.close(fig)

    fig, ax = plt.subplots(figsize=(3.45, 2.3))
    xk = np.arange(len(ATT))
    for j, (key, nm, col) in enumerate([("AUROC_TS", "$D$ / $TS$ (ours)", SLOTS[0]), ("AUROC_maxprob", "$1-\\max p$ (confidence)", SLOTS[1])]):
        ms = [stat([r["gate"][c][key] for r in rs]) for c in ATT]
        ax.bar(xk + (j - 0.5) * 0.38, [m[0] for m in ms], 0.36, yerr=[m[1] for m in ms], color=col, label=nm,
               error_kw=dict(lw=0.6, capsize=1.2, ecolor=INK))
    ax.axhline(0.5, color=MUTED, ls="--", lw=0.8); ax.text(len(ATT) - 0.5, 0.51, "chance", fontsize=6.5, color=MUTED, ha="right")
    ax.set_xticks(xk, [short[c].replace("\n", " ") for c in ATT], fontsize=6, color=INK, rotation=40, ha="right"); ax.set_ylim(0, 1.05); ax.set_ylabel("AUROC", fontsize=8, color=INK)
    style(ax); ax.legend(frameon=False, fontsize=6.5, loc="upper left")
    fig.tight_layout(); fig.savefig(PAPER / "figures" / "auroc.pdf"); fig.savefig(PAPER / "figures" / "auroc.png", dpi=200); plt.close(fig)

# ======================================================================== macros + summary
lines = ["% generated by aggregate.py -- do not edit by hand",
         "\\makeatletter", "\\newcommand{\\res}[1]{\\@ifundefined{res@#1}{\\textbf{??}}{\\@nameuse{res@#1}}}"]
lines += [f"\\@namedef{{res@{k}}}{{{v}}}" for k, v in sorted(macros.items())]
lines += ["\\makeatother"]
(PAPER / "numbers.tex").write_text("\n".join(lines), encoding="utf-8")
(RES / f"summary_{a.arch}.json").write_text(json.dumps(macros, indent=2), encoding="utf-8")
print(f"aggregated: cifar10 seeds={len(runs['cifar10'])}, svhn runs={len(runs['svhn'])}, backdoor={'yes' if bd else 'no'}; "
      f"{len(macros)} macros")

## 3. Data
CIFAR-10 and SVHN are downloaded once with torchvision and cached as uint8 arrays (`data/*.npz`), the same format the CPU pilot used.

In [ ]:
import numpy as np, torchvision, os
DATA = f'{ROOT}/data'
if not os.path.exists(f'{DATA}/cifar10.npz'):
    tr = torchvision.datasets.CIFAR10('/content/raw', train=True, download=True)
    te = torchvision.datasets.CIFAR10('/content/raw', train=False, download=True)
    np.savez(f'{DATA}/cifar10.npz', x_train=tr.data, y_train=np.array(tr.targets, dtype=np.int64),
             x_test=te.data, y_test=np.array(te.targets, dtype=np.int64))
if not os.path.exists(f'{DATA}/svhn.npz'):
    tr = torchvision.datasets.SVHN('/content/raw', split='train', download=True)
    te = torchvision.datasets.SVHN('/content/raw', split='test', download=True)
    np.savez(f'{DATA}/svhn.npz', x_train=tr.data.transpose(0, 2, 3, 1), y_train=tr.labels.astype(np.int64),
             x_test=te.data.transpose(0, 2, 3, 1), y_test=te.labels.astype(np.int64))
for f in ('cifar10', 'svhn'):
    d = np.load(f'{DATA}/{f}.npz'); print(f, {k: v.shape for k, v in d.items()})

## 4. Training
Seeds vary initialisation, batch order and augmentation; the train/validation split is fixed. Finished checkpoints are skipped automatically.

In [ ]:
ARCH = 'resnet18'
jobs = [f'--dataset cifar10 --seed {s} --epochs 30' for s in (0, 1, 2)] + \
       [f'--dataset svhn --seed {s} --epochs 15' for s in (0, 1, 2)] + \
       [f'--dataset cifar10 --poison {p} --epochs 30' for p in (0.01, 0.05, 0.10)]
for j in jobs:
    !python -u train_job.py {j} --arch {ARCH} --device cuda --data {DATA} 2>&1 | grep -v -i warn | tail -3

## 5. Evaluation (`--full`: 5,000 test images, 1,000 per expensive attack, full AutoAttack iterations)
Latency is measured once (seed 0). Each run writes `results/<dataset>_resnet18_s<seed>.json`.

In [ ]:
import os
for ds, seeds in (('cifar10', (0, 1, 2)), ('svhn', (0, 1, 2))):
    for s in seeds:
        if os.path.exists(f'results/{ds}_{ARCH}_s{s}.json'):
            print('done:', ds, s); continue
        lat = '--latency' if (ds, s) == ('cifar10', 0) else ''
        !python -u pub_eval.py --dataset {ds} --seed {s} --arch {ARCH} --device cuda --full {lat} --data {DATA} 2>&1 | grep -v -i warn | tail -12

## 6. Backdoor track

In [ ]:
if not os.path.exists(f'results/backdoor_{ARCH}.json'):
    !python -u pub_backdoor.py --arch {ARCH} --device cuda --data {DATA} 2>&1 | grep -v -i warn | tail -6

## 7. Tables, figures and number macros
Copy `paper/tables`, `paper/figures` and `paper/numbers.tex` into the paper folder (or compile there). `results/summary_resnet18.json` lists every reported number.

In [ ]:
!python aggregate.py --arch {ARCH}
!cd {ROOT}/08_Publication && zip -qr /content/lcd_publication_results.zip results paper logs
from google.colab import files; files.download('/content/lcd_publication_results.zip')